# Standout module — the pattern proposer: which measurement next?

*Executed notebook of the standout module (27 September 2026). The plan of record is the pre-registration of 26 September
(`GoalGathering/notes/PreRegistration_2026-09-26_Standout_Pattern_Proposer.md`) with its dated amendments and outcome sections; the design note is
`../DESIGN_2026-09-27.md`, the file map `../README.md`.*

**The task.** Every molecule of plan 05 has a coupling table Δ₂ that we can only afford to know in part: each entry costs an expensive energy. The recovery
reconstructs the table from *pattern responses* R_s = ½ aᵀ Δ a and stops when its held-out residual ρ_off is low enough. The deck fixes which patterns
exist and in which order they are measured. This module asks whether a network that has only seen other molecules can choose a better order — and,
since 27 September, whether the deck should reach every pair at all. The generative object is the order; the generator is a scorer of pairs of modes.

**Why it is a simulation with no new quantum chemistry.** The corpus's DFT-against-DFT Hessians give the full Δ₂ for 289 molecules, hence the exact
response of any pattern; ordering the deck and replaying the recovery costs CPU minutes per molecule. Every number is proxy-level (DFT against DFT) and is
labelled so.

In [1]:
import json, os, sys, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, str((Path("..") / ".." / "05_support_predictor" / "m05").resolve()))
import torch
torch.set_num_threads(int(os.environ.get("PP_THREADS", "1")))
from pp import core as C, scorer as S, embed_scorer as E
OUT = Path("..") / "out"
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
print("torch", torch.__version__, "| threads", torch.get_num_threads(), "| exports:", (OUT / "exports").exists())

torch 2.14.0+cpu | threads 1 | exports: True


*Reading this without chemistry.* The expensive calculations that anchor this project can be done in pieces: each piece (a 'pattern') costs one energy evaluation and reveals part of the table of couplings between vibrations. The question here is which pieces to measure first, so that the table is known after as few pieces as possible. 'Proxy level' means the rehearsal uses two cheap calculations standing in for the cheap and the expensive one; the test on real expensive responses is in sections 4c–4g. Terms are defined in `../../GLOSSARY.md`.

## 1. Data — the corpus Δ₂ response records and the splits

One export per molecule (`run_export.py`): modes and frequencies, atomic participations, the low-level Hessian (what a scorer may see), Δ₂ (the answer,
used only for the oracle and the truth-based read-outs), the probe deck's patterns and their exact responses. Splits are hashed and were fixed before the
first run: the parents (unsubstituted aromatics, larger than any training molecule) are evaluation only; A2/B molecules 70/10/20.

In [2]:
index = json.load(open(OUT / "exports" / "index.json"))["molecules"]
df = pd.DataFrame(index)
tab = df.groupby(["layer", "split"]).size().unstack(fill_value=0)
display(tab)
print(f"{len(df)} molecules; evaluation {int((df.split != 'train').sum() - (df.split == 'val').sum())} (parents {(df.split == 'eval_parents').sum()}), validation {(df.split == 'val').sum()}, training {(df.split == 'train').sum()}")
share = pd.DataFrame(json.load(open(OUT / "inband_share_2026-09-26.json")))
print("in-band share of off-diagonal coupling power (26 Sep), mean per layer:", share.groupby("layer")["inband"].mean().round(3).to_dict(), "| all:", round(float(share.inband.mean()), 3))
fig, ax = plt.subplots(figsize=(5, 3)); ax.hist(share.inband, bins=20); ax.set_xlabel("share of off-diagonal coupling power inside the 200 cm⁻¹ band"); ax.set_ylabel("molecules")
fig.tight_layout(); fig.savefig(FIG / "inband_share.png", dpi=120); plt.show()

split,eval,eval_parents,train,val
layer,,,,
A,0,41,0,0
A2,40,0,131,15
B,16,0,40,6


289 molecules; evaluation 97 (parents 41), validation 21, training 171
in-band share of off-diagonal coupling power (26 Sep), mean per layer: {'A': 0.411, 'A2': 0.465, 'B': 0.435} | all: 0.451


C:\Users\thebr\AppData\Local\Temp\ipykernel_29800\244008003.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "inband_share.png", dpi=120); plt.show()


## 2. The recovery, replayed live on one small molecule

Consume the single-mode block, then the off-diagonal patterns in the given order; after every stride solve the banded-ℓ₁ recovery (FISTA, warm-started,
λ chosen on the held-out patterns) and record the held-out ρ_off — the plan's own stopping quantity, which needs no truth. Three orders: P0 (the deck's
hashed order), P1 (the hand-feature scorer, seed 0) and P3 (the oracle, which knows Δ₂).

In [3]:
ids = sorted([r["id"] for r in index if r["id"].startswith("B_") and r["split"] != "train"], key=lambda i: next(x for x in index if x["id"] == i).get("M", 0))
mid = ids[0] if ids else sorted(r["id"] for r in index if r["id"].startswith("B_"))[0]
e = C.load_export(OUT / "exports" / f"{mid}.npz")
X, _, pairs = S.pair_features(e)
sc = S.Scorer.load(OUT / "p1", 0)
pool = C.order_p0(e); stride = max(2, int(np.ceil(len(pool) / 30)))
t0 = time.time()
curves = {"P0 (hashed)": C.rho_curve(e, pool, stride=stride),
          "P1 (hand-feature scorer)": C.rho_curve(e, C.order_by_scores(e, S.scores_matrix(e, sc.predict(X), pairs)), stride=stride),
          "P3 (oracle)": C.rho_curve(e, C.order_oracle(e), stride=stride)}
print(f"{mid}: M = {e['M']} modes, {len(pool)} off-diagonal patterns in the band pool, stride {stride}; {time.time() - t0:.0f} s for three curves")
fig, ax = plt.subplots(figsize=(6, 3.6))
for name, c in curves.items():
    ax.plot([r[0] for r in c], [r[2] for r in c], label=name)
ax.set_xlabel("energies consumed"); ax.set_ylabel("held-out ρ_off"); ax.set_ylim(0, 1.05); ax.legend(); ax.set_title(f"{mid}: the recovery under three orders")
fig.tight_layout(); fig.savefig(FIG / "one_molecule_curves.png", dpi=120); plt.show()
for name, c in curves.items():
    print(f"  {name:26s} ρ_off after the single block {c[0][2]:.3f}, at the end {c[-1][2]:.3f}; energies to ρ_off ≤ 0.3: {C.k_off_at(c, 0.3, e['M'])}")

B_056dec31a6: M = 27 modes, 178 off-diagonal patterns in the band pool, stride 6; 6 s for three curves
  P0 (hashed)                ρ_off after the single block 1.000, at the end 0.495; energies to ρ_off ≤ 0.3: None
  P1 (hand-feature scorer)   ρ_off after the single block 1.000, at the end 0.496; energies to ρ_off ≤ 0.3: None
  P3 (oracle)                ρ_off after the single block 1.000, at the end 0.495; energies to ρ_off ≤ 0.3: None


C:\Users\thebr\AppData\Local\Temp\ipykernel_29800\500612809.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "one_molecule_curves.png", dpi=120); plt.show()


## 3. The orderings, and the fair chance for the learned representation

**P1** — an MLP on 21 hand-made pair features (frequencies, |Δω|, the band flag, atom-sharing overlaps of the participations, projections of the low-level
Hessian) → log₁₀|Δ_ij|. **P2** — the rung-C equivariant body of module 05 (unchanged) turns atoms, coordinates and the low-level Hessian into per-atom features;
a mode's embedding is its participation-weighted sum; a pair head reads two embeddings and |Δω|. **P12** — the z-score average of the two. A pattern's score is
the *mean* over the pairs it touches (dated amendment: the sum favoured broad random patterns on the planted test).

The user's rule of 26 September 12:1x: no sentence about the learned representation before a five-stage search (recipe, loss, capacity, data growth,
pretraining), each stage read on the validation split by the ordering metric with a margin fixed before the numbers.

In [4]:
rows = []
for f, stage in ((OUT / "stage2_readout_2026-09-26.json", "stage 2"), (OUT / "stage3_readout_2026-09-26.json", "stage 3")):
    if f.exists():
        r = json.load(open(f))
        for name, res in r["results"].items():
            s = res["summary"]
            rows.append(dict(read=stage, recipe=name, spearman=round(s["spearman"]["mean"], 4), spearman_min=round(s["spearman"]["min"], 4), spearman_max=round(s["spearman"]["max"], 4),
                             mse=round(s["mse"]["mean"], 4), p10_out_of_band=round(s["p10_out"]["mean"], 3), p10_in_band=round(s["p10_in"]["mean"], 3)))
val = pd.DataFrame(rows).drop_duplicates(subset=["recipe"], keep="last")
display(val)
p2 = E.EmbedScorer.load(OUT / "p2s1_lr1e-3_w128", 0)
print("P2 (stage-1 recipe): parameters", sum(p.numel() for p in p2.params), "| P1 features:", X.shape[1])
print("Reading rule: stage 2 by the incumbent's seed spread (18:1x), stage 3 the same; from stage 4 a paired per-molecule Wilcoxon test. Stages 1–3: recipe +0.03, loss and capacity within the margins; nothing licensed either way.")

,read,recipe,spearman,spearman_min,spearman_max,mse,p10_out_of_band,p10_in_band
1,stage 2,p2,0.6110,0.6089,0.6133,0.3461,0.426,0.358
3,stage 2,p2s2_huber,0.6382,0.6308,0.6452,0.3226,0.472,0.396
4,stage 2,p2s2_rank,0.6397,0.6358,0.6418,0.3338,0.468,0.384
5,stage 3,P1,0.5821,0.5804,0.5844,0.3807,0.432,0.390
6,stage 3,p2s1_lr1e-3_w128,0.6362,0.6237,0.6443,0.3263,0.470,0.377
7,stage 3,p2s3_b5,0.6425,0.6400,0.6441,0.3191,0.475,0.404
8,stage 3,p2s3_b3v,0.6422,0.6368,0.6479,0.3201,0.477,0.396
9,stage 3,p2s3_b5v,0.6478,0.6438,0.6506,0.3154,0.479,0.411


P2 (stage-1 recipe): parameters 230307 | P1 features: 21
Reading rule: stage 2 by the incumbent's seed spread (18:1x), stage 3 the same; from stage 4 a paired per-molecule Wilcoxon test. Stages 1–3: recipe +0.03, loss and capacity within the margins; nothing licensed either way.


## 4. The registered read-outs — E1 (band pool), E2 (all pairs), the adaptive variants

Rebuilt from the committed read-out JSONs (`out/sim/*_readout.json`, produced by `readout.py` from the pools that ran on hel1-23). Primary read-out:
K_off(0.3) as a ratio to P0; fallback where P0 does not reach 0.3 (the band pool): n_half (energies to the midpoint between the single block and P0's end
point) and the AUC of ρ_off over P0's checkpoint grid, both as per-molecule ratios (median; fraction improved).

In [5]:
def table(readout, split, tag="rho_off"):
    r = json.load(open(OUT / "sim" / readout))[split]
    t = r[tag]
    rows = [dict(ordering=n, n=v["n_half"], median_n_half_ratio=v["median_half_ratio"], half_improved=v["frac_half_improved"], median_auc_ratio=v["median_auc_ratio"], auc_improved=v["frac_auc_improved"])
            for n, v in t.items() if isinstance(v, dict)]
    d = pd.DataFrame(rows).set_index("ordering").round(3)
    d.attrs["P0"] = (t["P0_start_median"], t["P0_final_median"], r["n_molecules"], r["P0_reaches_K_off_0p3"])
    return d
summary_rows = []
for readout, label in (("band_p2_readout.json", "E1 band pool"), ("all_p2_readout.json", "E2 all pairs"), ("band_p2s1A_readout.json", "band pool + adaptive")):
    for split in ("eval_parents", "eval"):
        d = table(readout, split)
        s, f, n, k = d.attrs["P0"]
        display(Markdown(f"**{label}, {split} ({n} molecules): P0 median ρ_off {s:.3f} → {f:.3f} over the whole deck; P0 reaches 0.3 on {k}**"))
        display(d)
        for name in d.index:
            summary_rows.append(dict(experiment=label, split=split, ordering=name, median_n_half_ratio=d.loc[name, "median_n_half_ratio"]))
sm = pd.DataFrame(summary_rows)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, (label, g) in zip(axes, sm.groupby("experiment", sort=False)):
    piv = g.pivot(index="ordering", columns="split", values="median_n_half_ratio")
    piv.plot.barh(ax=ax, legend=(ax is axes[0])); ax.set_title(label); ax.set_xlabel("median n_half ratio vs P0"); ax.axvline(1.0, color="k", lw=0.8)
fig.tight_layout(); fig.savefig(FIG / "n_half_ratios.png", dpi=120); plt.show()

**E1 band pool, eval_parents (41 molecules): P0 median ρ_off 1.000 → 0.530 over the whole deck; P0 reaches 0.3 on 1**

,n,median_n_half_ratio,half_improved,median_auc_ratio,auc_improved
ordering,,,,,
P12_seed0,41,0.171,0.951,0.849,0.878
P12_seed1,41,0.200,0.927,0.829,0.878
P12_seed2,41,0.160,0.951,0.844,0.902
P1_seed0,41,0.175,0.951,0.847,0.854
P1_seed1,41,0.200,0.951,0.869,0.854
P1_seed2,41,0.185,0.951,0.860,0.878
P2_seed0,41,0.200,0.927,0.860,0.878
P2_seed1,41,0.214,0.927,0.877,0.829
P2_seed2,41,0.206,0.878,0.853,0.878


**E1 band pool, eval (56 molecules): P0 median ρ_off 1.000 → 0.471 over the whole deck; P0 reaches 0.3 on 3**

,n,median_n_half_ratio,half_improved,median_auc_ratio,auc_improved
ordering,,,,,
P12_seed0,56,0.174,0.982,0.815,0.929
P12_seed1,56,0.167,0.982,0.816,0.929
P12_seed2,56,0.167,0.982,0.823,0.929
P1_seed0,56,0.200,0.982,0.825,0.946
P1_seed1,56,0.169,0.964,0.839,0.929
P1_seed2,56,0.174,0.982,0.830,0.893
P2_seed0,56,0.203,0.982,0.841,0.911
P2_seed1,56,0.253,0.946,0.853,0.929
P2_seed2,56,0.203,0.946,0.838,0.911


**E2 all pairs, eval_parents (41 molecules): P0 median ρ_off 1.000 → 0.076 over the whole deck; P0 reaches 0.3 on 38**

...

**band pool + adaptive, eval_parents (41 molecules): P0 median ρ_off 1.000 → 0.530 over the whole deck; P0 reaches 0.3 on 1**

,n,median_n_half_ratio,half_improved,median_auc_ratio,auc_improved
ordering,,,,,
P0A,41,0.824,0.756,0.918,0.951
P12_seed0,41,0.143,0.951,0.844,0.878
P12_seed1,41,0.162,0.927,0.855,0.878
P12_seed2,41,0.174,0.951,0.831,0.902
P1A_seed0,41,0.176,0.976,0.791,0.976
P1A_seed1,41,0.200,0.976,0.796,1.000
P1A_seed2,41,0.174,0.951,0.764,0.976
P1_seed0,41,0.175,0.951,0.847,0.854
P1_seed1,41,0.200,0.951,0.869,0.854


**band pool + adaptive, eval (56 molecules): P0 median ρ_off 1.000 → 0.471 over the whole deck; P0 reaches 0.3 on 3**

,n,median_n_half_ratio,half_improved,median_auc_ratio,auc_improved
ordering,,,,,
P0A,56,0.791,0.893,0.926,0.964
P12_seed0,56,0.172,0.982,0.811,0.929
P12_seed1,56,0.150,0.982,0.813,0.911
P12_seed2,56,0.185,0.982,0.819,0.911
P1A_seed0,56,0.200,0.982,0.769,0.982
P1A_seed1,56,0.161,0.964,0.778,0.982
P1A_seed2,56,0.169,0.964,0.776,0.964
P1_seed0,56,0.200,0.982,0.825,0.946
P1_seed1,56,0.169,0.964,0.839,0.929


C:\Users\thebr\AppData\Local\Temp\ipykernel_29800\4151270001.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "n_half_ratios.png", dpi=120); plt.show()


### 4a. Adaptive ordering, paired per molecule (S5)

P0+A, P1+A, P2+A re-rank the remaining patterns after every checkpoint by z(prior) + z(feedback), feedback being the mean over touched pairs of
log₁₀|Δ̂_ij| from the current reconstruction, with optimism for untouched pairs (the correction the planted dry run forced before any real molecule ran).
The paired statistics below are computed from the merged simulation JSON when it is present locally and otherwise loaded from the committed
`out/sim/band_p2s1A_paired.json` written by this cell.

In [6]:
sys.path.insert(0, str(Path("..").resolve()))
from readout import per_molecule
paired_path = OUT / "sim" / "band_p2s1A_paired.json"
merged = OUT / "sim" / "band_p2s1A_merged.json"
if merged.exists():
    res = json.load(open(merged))
    pairs_ab = [("P0A", "P0")] + [(f"P{k}A_seed{s}", f"P{k}_seed{s}") for k in (1, 2) for s in range(3)]
    paired = {}
    for split in ("eval_parents", "eval"):
        mols = {i: m for i, m in res["per_molecule"].items() if m["split"] == split}
        paired[split] = {}
        for a, b in pairs_ab:
            nh, au = [], []
            for m in mols.values():
                pm = per_molecule(m, [a] if a == "P0A" else [a, b])
                x = pm[(a, "rho_off")]
                yh = pm[(a, "rho_off")]["base_half"] if a == "P0A" else pm[(b, "rho_off")]["n_half"]
                ya = 1.0 if a == "P0A" else pm[(b, "rho_off")]["auc_ratio"]
                if x["n_half"] is not None and yh is not None: nh.append((x["n_half"], yh))
                au.append((x["auc_ratio"], ya))
            paired[split][f"{a} vs {b}"] = dict(n=len(mols), n_half_better=float(np.mean([u < v for u, v in nh])), n_half_equal=float(np.mean([u == v for u, v in nh])),
                                              median_n_half_ratio=float(np.median([u / max(v, 1) for u, v in nh])), auc_better=float(np.mean([u < v for u, v in au])),
                                              median_auc_ratio=float(np.median([u / v for u, v in au])))
    json.dump(paired, open(paired_path, "w"), indent=1)
else:
    paired = json.load(open(paired_path))
for split, t in paired.items():
    display(Markdown(f"**{split}** — fraction of molecules where the adaptive variant is strictly better / equal, and median ratios adaptive/fixed"))
    display(pd.DataFrame(t).T.round(3))
print("S5 (P1+A better than P1 on ≥ 70 % of molecules): fails on n_half (mostly ties at the checkpoint grid), passes on the AUC (95–100 %). The oracle gap is knowledge, not feedback.")

**eval_parents** — fraction of molecules where the adaptive variant is strictly better / equal, and median ratios adaptive/fixed

,n,n_half_better,n_half_equal,median_n_half_ratio,auc_better,median_auc_ratio
P0A vs P0,41.0,0.707,0.073,0.824,0.951,0.918
P1A_seed0 vs P1_seed0,41.0,0.244,0.634,1.000,1.000,0.909
P1A_seed1 vs P1_seed1,41.0,0.293,0.561,1.000,0.951,0.906
P1A_seed2 vs P1_seed2,41.0,0.390,0.512,1.000,0.976,0.898
P2A_seed0 vs P2_seed0,41.0,0.268,0.659,1.000,1.000,0.921
P2A_seed1 vs P2_seed1,41.0,0.317,0.659,1.000,0.976,0.924
P2A_seed2 vs P2_seed2,41.0,0.317,0.659,1.000,0.951,0.911


**eval** — fraction of molecules where the adaptive variant is strictly better / equal, and median ratios adaptive/fixed

,n,n_half_better,n_half_equal,median_n_half_ratio,auc_better,median_auc_ratio
P0A vs P0,56.0,0.893,0.018,0.791,0.964,0.926
P1A_seed0 vs P1_seed0,56.0,0.125,0.786,1.000,0.964,0.936
P1A_seed1 vs P1_seed1,56.0,0.250,0.679,1.000,0.982,0.933
P1A_seed2 vs P1_seed2,56.0,0.214,0.696,1.000,0.964,0.931
P2A_seed0 vs P2_seed0,56.0,0.357,0.589,1.000,0.982,0.945
P2A_seed1 vs P2_seed1,56.0,0.286,0.661,1.000,0.982,0.935
P2A_seed2 vs P2_seed2,56.0,0.250,0.696,1.000,0.982,0.941


S5 (P1+A better than P1 on ≥ 70 % of molecules): fails on n_half (mostly ties at the checkpoint grid), passes on the AUC (95–100 %). The oracle gap is knowledge, not feedback.


### 4b. Follow-up (28 September 2026): the same paired test on the wide (all-pairs) pool — `all_p2s1A`

Section 4a read the adaptive variants on the band pool. The wide pool of section 5 is the candidate set the plan will use, so the adaptive
orderings were run there too (hel1-23, 8 shards, 27 Sep 13:2x → 28 Sep 06:39 UTC; the same 97 evaluation molecules; merged record
`out/sim/all_p2s1A_merged.json`, local). The paired statistics are computed from the merged record when it is present and otherwise loaded from the
committed `out/sim/all_p2s1A_paired.json` this cell writes. Two lines were fixed before the run: S5 (P1+A better than P1 on ≥ 70 % of the parents)
and the prediction of 27 Sep 18:4x that feedback closes at least a third of the log-gap from P1 to the oracle on n_half.

In [7]:
paired_path_w = OUT / "sim" / "all_p2s1A_paired.json"
merged_w = OUT / "sim" / "all_p2s1A_merged.json"
if merged_w.exists():
    resw = json.load(open(merged_w))
    pairs_ab = [("P0A", "P0")] + [(f"P{k}A_seed{s}", f"P{k}_seed{s}") for k in (1, 2) for s in range(3)]
    paired_w = {}
    for split in ("eval_parents", "eval"):
        mols = {i: m for i, m in resw["per_molecule"].items() if m["split"] == split}
        paired_w[split] = {}
        for a, b in pairs_ab:
            nh, au = [], []
            for m in mols.values():
                pm = per_molecule(m, [a] if a == "P0A" else [a, b])
                x = pm[(a, "rho_off")]
                yh = pm[(a, "rho_off")]["base_half"] if a == "P0A" else pm[(b, "rho_off")]["n_half"]
                ya = 1.0 if a == "P0A" else pm[(b, "rho_off")]["auc_ratio"]
                if x["n_half"] is not None and yh is not None: nh.append((x["n_half"], yh))
                au.append((x["auc_ratio"], ya))
            paired_w[split][f"{a} vs {b}"] = dict(n=len(mols), n_half_better=float(np.mean([u < v for u, v in nh])), n_half_equal=float(np.mean([u == v for u, v in nh])),
                                                median_n_half_ratio=float(np.median([u / max(v, 1) for u, v in nh])), auc_better=float(np.mean([u < v for u, v in au])),
                                                median_auc_ratio=float(np.median([u / v for u, v in au])))
        med = {}
        for name in ("P1_seed0", "P1A_seed0", "P3_oracle"):                      # the prediction line: seed-0 medians of the n_half ratio against P0 (as in the read-out of 28 Sep 08:4x)
            rat = []
            for m in mols.values():
                r_ = per_molecule(m, [name])[(name, "rho_off")]
                if r_["n_half"] is not None and r_["base_half"]: rat.append(r_["n_half"] / r_["base_half"])
            med[name] = float(np.median(rat))
        paired_w[split]["log_gap_closure_seed0"] = dict(median_n_half_ratio_vs_P0=med, closure=float((np.log(med["P1_seed0"]) - np.log(med["P1A_seed0"])) /
                                                                                    (np.log(med["P1_seed0"]) - np.log(med["P3_oracle"]))))
    json.dump(paired_w, open(paired_path_w, "w"), indent=1)
else:
    paired_w = json.load(open(paired_path_w))
for split, t in paired_w.items():
    lg = t["log_gap_closure_seed0"]; tt = {k: v for k, v in t.items() if k != "log_gap_closure_seed0"}
    display(Markdown(f"**{split}**, wide pool — adaptive against fixed, paired per molecule (same seed); median n_half ratio against P0, seed 0: "
                     f"P1 {lg['median_n_half_ratio_vs_P0']['P1_seed0']:.2f}, P1+A {lg['median_n_half_ratio_vs_P0']['P1A_seed0']:.2f}, oracle {lg['median_n_half_ratio_vs_P0']['P3_oracle']:.2f} → "
                     f"feedback closes {100 * lg['closure']:.0f} % of the log-gap P1 → oracle"))
    display(pd.DataFrame(tt).T.round(3))
p1 = [paired_w["eval_parents"][f"P1A_seed{s} vs P1_seed{s}"]["n_half_better"] for s in range(3)]
cl = paired_w["eval_parents"]["log_gap_closure_seed0"]["closure"]
print(f"S5 on the wide pool, n_half of ρ_off (P1+A better than P1 on ≥ 70 % of the parents): {'PASS' if min(p1) >= 0.7 else 'FAIL'} ({100 * min(p1):.0f}–{100 * max(p1):.0f} %; "
      f"on the K_off(0.3) read-out of `paired_readout.py`, out/sim/all_p2s1A_paired.md: 42–50 %); "
      f"prediction of 27 Sep 18:4x (feedback closes ≥ a third of the log-gap P1 → oracle): {'PASS' if cl >= 1 / 3 else 'FAIL'} ({100 * cl:.0f} %)")

**eval_parents**, wide pool — adaptive against fixed, paired per molecule (same seed); median n_half ratio against P0, seed 0: P1 0.57, P1+A 0.50, oracle 0.18 → feedback closes 12 % of the log-gap P1 → oracle

,n,n_half_better,n_half_equal,median_n_half_ratio,auc_better,median_auc_ratio
P0A vs P0,41.0,0.780,0.049,0.727,0.854,0.921
P1A_seed0 vs P1_seed0,41.0,0.293,0.512,1.000,0.585,0.994
P1A_seed1 vs P1_seed1,41.0,0.268,0.439,1.000,0.585,0.990
P1A_seed2 vs P1_seed2,41.0,0.244,0.561,1.000,0.585,0.983
P2A_seed0 vs P2_seed0,41.0,0.268,0.561,1.000,0.390,1.004
P2A_seed1 vs P2_seed1,41.0,0.293,0.561,1.000,0.463,1.007
P2A_seed2 vs P2_seed2,41.0,0.244,0.683,1.000,0.610,0.996


**eval**, wide pool — adaptive against fixed, paired per molecule (same seed); median n_half ratio against P0, seed 0: P1 0.43, P1+A 0.44, oracle 0.20 → feedback closes -4 % of the log-gap P1 → oracle

,n,n_half_better,n_half_equal,median_n_half_ratio,auc_better,median_auc_ratio
P0A vs P0,56.0,0.750,0.071,0.721,0.571,0.991
P1A_seed0 vs P1_seed0,56.0,0.125,0.643,1.000,0.250,1.006
P1A_seed1 vs P1_seed1,56.0,0.107,0.696,1.000,0.357,1.007
P1A_seed2 vs P1_seed2,56.0,0.089,0.750,1.000,0.286,1.006
P2A_seed0 vs P2_seed0,56.0,0.071,0.732,1.000,0.321,1.008
P2A_seed1 vs P2_seed1,56.0,0.125,0.732,1.000,0.411,1.002
P2A_seed2 vs P2_seed2,56.0,0.089,0.714,1.000,0.268,1.007


S5 on the wide pool, n_half of ρ_off (P1+A better than P1 on ≥ 70 % of the parents): FAIL (24–29 %; on the K_off(0.3) read-out of `paired_readout.py`, out/sim/all_p2s1A_paired.md: 42–50 %); prediction of 27 Sep 18:4x (feedback closes ≥ a third of the log-gap P1 → oracle): FAIL (12 %)


*Reading (28 September 2026).* Feedback alone helps the blind order: P0+A against P0 0.73 / 0.72 on n_half (parents / substituted), and on the registered
K_off(0.3) read-out 0.92 on the parents and 1.08 on the substituted molecules — the gain sits early in the curve, not at the tight end. On top of a scorer it adds nothing on the wide pool — P1+A and P2+A tie their fixed counterparts at the median (on n_half here, and on the
registered K_off(0.3) read-out in `out/sim/all_p2s1A_paired.md`) and the AUC gain of the band pool is gone. What the adaptive loop can learn from the reconstruction, the learned scorer already knows before the first measurement; the remaining
factor of about 2.5–3 to the oracle is knowledge of the molecule. The lever for the module and for plan 05 is a better scorer (more molecules,
CC-level responses), not an adaptive campaign; the adaptive variants stay in the code as a measured negative. Section 4a stands as run.

### 4c. Follow-up (28 September 2026, later): the plan on a real coupled-cluster response — benzene (pre-registration `PreRegistration_2026-09-28_Standout_CC_Level_Test.md`)

Everything above is proxy-level (ωB97X − B3LYP). Benzene has a CCSD(T)/cc-pVDZ Hessian at the corpus geometry (E8, 24 September). The same export code
builds the CC response record (`pp.core.export_molecule(..., hi_override=…)`: same B3LYP modes, same deck and hash), the same recovery runs, and the
registered lines C1–C3 are read: `cc_level_test.py` → `out/cc/A_8448043181_cc_test.json`. Two exploratory runs with the wide pool are read beside it, labelled.

In [8]:
cc_reg = json.load(open(OUT / "cc" / "A_8448043181_cc_test.json")); cc_e1 = json.load(open(OUT / "cc" / "A_8448043181_cc_test_all_band200.json")); cc_e2 = json.load(open(OUT / "cc" / "A_8448043181_cc_test_all_band5000.json"))
rows = []
for tag, rr in (("registered: band deck, band prior", cc_reg), ("exploratory: wide pool, band prior", cc_e1), ("exploratory: wide pool, open prior", cc_e2)):
    for resp in ("cc", "proxy"):
        d = rr[resp]
        rows.append({"run": tag, "response": resp, "P0 K_off(0.3)": d["P0"]["k_off_0p3"], "P0 final ρ_off": round(d["P0"]["rho_off_final"], 2), "P1 median K_off ratio": d["P1_median"]["k_off_ratio"],
                     "P1 median n_half ratio": d["P1_median"]["n_half_ratio"], "oracle K_off(0.3)": d["oracle"]["k_off_0p3"]})
display(pd.DataFrame(rows))
print(f"in-band share of the off-diagonal Δ₂ power (200 cm⁻¹): CC {cc_reg['delta2']['inband_share_cc']:.2f} vs proxy {cc_reg['delta2']['inband_share_proxy']:.2f}; "
      f"registered lines: C1 {'pass' if cc_reg['judged']['C1_pass'] else 'FAIL'}, C2 {'pass' if cc_reg['judged']['C2_pass'] else 'FAIL'}; label for the plan line: {cc_reg['judged']['label']}")

,run,response,P0 K_off(0.3),P0 final ρ_off,P1 median K_off ratio,P1 median n_half ratio,oracle K_off(0.3)
0,"registered: band deck, band prior",cc,NaN,0.78,NaN,1.080000,NaN
1,"registered: band deck, band prior",proxy,468.0,0.24,1.000000,2.000000,36.0
2,"exploratory: wide pool, band prior",cc,1116.0,0.01,1.166667,0.444444,434.0
3,"exploratory: wide pool, band prior",proxy,496.0,0.00,1.750000,3.666667,124.0
4,"exploratory: wide pool, open prior",cc,1054.0,0.00,1.176471,0.230769,372.0
5,"exploratory: wide pool, open prior",proxy,310.0,0.00,2.200000,2.750000,124.0


in-band share of the off-diagonal Δ₂ power (200 cm⁻¹): CC 0.04 vs proxy 0.52; registered lines: C1 FAIL, C2 FAIL; label for the plan line: proxy (C1 failed on CC)


*Reading (28 September 2026).* On the real correction the band the whole deck is built on misses the couplings: 4 % of the off-diagonal power lies within
200 cm⁻¹ (proxy: 52 %), most of it sits between same-symmetry modes 300–1,000 cm⁻¹ apart, and no ordering — the oracle included — reaches the target on the
band deck. With the wide pool the oracle reaches it; the band-trained scorer does not beat the hashed order there on this one molecule. The band-prior finding of
26 September was a proxy statement; the wide-candidate deck pre-registered on 27 September is the condition for reaching the target, not a refinement; the
prediction on record (C1 passes) was wrong and is kept. Sections 1–4b stand as run; naphthalene's CC Hessian (this week) decides whether benzene is the rule.

### 4d. Follow-up (29 September 2026): benzene's corpus psi4 pair is noise — the CC test rerun with the analytic B3LYP low level

While building the anchor-set-two read-out it turned out that benzene's corpus row (psi4 finite-difference Hessians, grid 75/302) is off the pyscf
analytic Hessians of the same geometry by up to 132 cm⁻¹ (ωB97X) and 23 cm⁻¹ (B3LYP): the proxy Δ₂ of that row is noise, and 4c's Δ_CC carried the
B3LYP file's error. Other rows checked (benzonitrile, fluorobenzene) agree with their analytic pairs to 0–5 cm⁻¹. Section 4c is kept as run; here the
same test with `--use-analytic` (analytic low level, analytic proxy export, same deck and hash). Correction section in the pre-registration.

In [9]:
an_reg = json.load(open(OUT / "cc" / "A_8448043181_cc_test_analytic.json")); an_e = json.load(open(OUT / "cc" / "A_8448043181_cc_test_analytic_all_band0.json"))
old_e = json.load(open(OUT / "cc" / "A_8448043181_cc_test_all_band0.json")) if (OUT / "cc" / "A_8448043181_cc_test_all_band0.json").exists() else cc_e2
rows = []
for tag, rr in (("28 Sep, psi4 FD low level: registered band deck", cc_reg), ("29 Sep, analytic low level: registered band deck", an_reg),
                ("28 Sep, psi4 FD low level: wide pool, open prior", old_e), ("29 Sep, analytic low level: wide pool, open prior", an_e)):
    for resp in ("cc", "proxy"):
        d = rr[resp]
        rows.append({"run": tag, "response": resp, "P0 K_off(0.3)": d["P0"]["k_off_0p3"], "P0 final ρ_off": round(d["P0"]["rho_off_final"], 2), "P1 median K_off ratio": d["P1_median"]["k_off_ratio"],
                     "P1 median n_half ratio": d["P1_median"]["n_half_ratio"], "oracle K_off(0.3)": d["oracle"]["k_off_0p3"]})
display(pd.DataFrame(rows))
print(f"in-band share of the off-diagonal Δ₂ power (200 cm⁻¹): CC {an_reg['delta2']['inband_share_cc']:.2f} (4c: {cc_reg['delta2']['inband_share_cc']:.2f}) vs proxy "
      f"{an_reg['delta2']['inband_share_proxy']:.2f} (4c: {cc_reg['delta2']['inband_share_proxy']:.2f}); Frobenius CC/proxy {an_reg['delta2']['ratio']:.2f} (4c: {cc_reg['delta2']['ratio']:.2f}); "
      f"registered lines: C1 {'pass' if an_reg['judged']['C1_pass'] else 'FAIL'}, C2 {'pass' if an_reg['judged']['C2_pass'] else 'FAIL'}; "
      f"wide pool, open prior, analytic: C1 {'pass' if an_e['judged']['C1_pass'] else 'FAIL'}, C2 {'pass' if an_e['judged']['C2_pass'] else 'FAIL'} ({an_e['judged']['label']})")

,run,response,P0 K_off(0.3),P0 final ρ_off,P1 median K_off ratio,P1 median n_half ratio,oracle K_off(0.3)
0,"28 Sep, psi4 FD low level: registered band deck",cc,NaN,0.78,NaN,1.080000,NaN
1,"28 Sep, psi4 FD low level: registered band deck",proxy,468.0,0.24,1.000000,2.000000,36.0
2,"29 Sep, analytic low level: registered band deck",cc,NaN,0.56,NaN,1.173913,NaN
3,"29 Sep, analytic low level: registered band deck",proxy,NaN,0.80,NaN,1.130435,NaN
4,"28 Sep, psi4 FD low level: wide pool, open prior",cc,558.0,0.02,1.444444,0.428571,372.0
5,"28 Sep, psi4 FD low level: wide pool, open prior",proxy,310.0,0.00,2.200000,3.666667,124.0
6,"29 Sep, analytic low level: wide pool, open prior",cc,806.0,0.01,0.461538,0.571429,124.0
7,"29 Sep, analytic low level: wide pool, open prior",proxy,620.0,0.06,0.700000,0.555556,124.0


in-band share of the off-diagonal Δ₂ power (200 cm⁻¹): CC 0.04 (4c: 0.04) vs proxy 0.30 (4c: 0.52); Frobenius CC/proxy 2.36 (4c: 0.33); registered lines: C1 FAIL, C2 FAIL; wide pool, open prior, analytic: C1 pass, C2 FAIL (plan works on CC, proxy misjudged its size)


*Reading (29 September 2026).* What stands from 4c: the CC correction has 4 % of its off-diagonal power in band and no ordering reaches ρ_off 0.3 on the
band deck; the registered lines fail. What is withdrawn: every proxy number for benzene — the clean proxy has 30 % in band (not 52), is 2.4× smaller off the
diagonal than the CC correction (not 3× larger), and fails the band deck just as CC does. What is new and exploratory: on the wide pool with the open prior
the band-trained scorer halves the cost on the real CC response (P1 median 0.46 of P0's 806 energies; oracle 124), where 4c had read "worse than P0" from
the noisy files. One molecule, one scorer; nothing about the learned representation changes. Rule for every CC-level read from here: the analytic pair as
the low level, never a corpus psi4 file. Naphthalene decides, read the same way.

### 4e. Follow-up (30 September 2026): the stop rule read again with the band-free solver prior (reading (ii))

Reading (i) of 28 September (section 4c's pre-registration amendment) failed both stop-rule lines under the band prior: 53 % of the 97 evaluation
molecules stopped within twice the band deck (line 90 %) and 29 % of the stops were false (line 5 %). The amendment named the band prior as one suspect
and registered reading (ii): the same recovery with the band-free solver prior (`--w-cm 0`), P12 and the oracle only, eight laptop shards
(29 Sep 21:57 → 22:58). Same read-out tool, same lines; the band record is used only for B_max, as registered.

In [10]:
r1 = json.load(open(OUT / "sim" / "stop_rule_band_prior_2026-09-28.json")); r2 = json.load(open(OUT / "sim" / "stop_rule_band_free_prior_2026-09-30.json"))
rows = []
for tag, r in (("reading (i), 28 Sep: band prior", r1), ("reading (ii), 30 Sep: band-free prior", r2)):
    j = r["judged"]
    rows.append({"reading": tag, "judged on": j["order"], "stopped within B_max": f"{100 * j['frac_stopped']:.0f} %", "cost ratio of medians": round(j["cost_ratio_of_medians"], 2),
                 "false stops": f"{100 * j['frac_false_stops']:.0f} %", "W1": "pass" if j["W1_pass"] else "FAIL", "W2": "pass" if j["W2_pass"] else "FAIL"})
display(pd.DataFrame(rows).set_index("reading"))

,judged on,stopped within B_max,cost ratio of medians,false stops,W1,W2
reading,,,,,,
"reading (i), 28 Sep: band prior",P12,53 %,1.06,29 %,FAIL,FAIL
"reading (ii), 30 Sep: band-free prior",P12,40 %,1.15,28 %,FAIL,FAIL


*Reading (30 September 2026).* The prior was not the problem: with the band-free prior P12 stops on 40 % (was 53 %) with 28 % false stops (was 29 %);
the oracle stops on 55 % with 2 % false stops at 0.54× the band deck. W1 and W2 fail again. What this settles is the diagnosis of the 28 September
amendment: the stop quantity reads a held-out set that belongs to the band deck and a threshold with no margin, and the budget is defined against the
band deck that is no longer the plan's reference. The next registration (`Draft_2026-09-29_Stop_Rule_Registration_Two.md`: held-out set drawn from the
wide pool, τ with a margin fitted on the validation split, B_max from the measured cost distribution) waits for the user's word; nothing in it was
fitted to these numbers.

### 4f. Follow-up (30 September 2026): the CC response itself was wrong until 29 September — the test on the corrected benzene Hessian

On 29 September the probe that produces the CC response (`e8_cc_hessian_fd.py`) turned out to call pyscf's CCSD(T) gradient without an explicit (T) lambda,
which silently uses the CCSD lambda: the gradient is not dE/dx of the CCSD(T) energy (water: 1.5e-3 a.u. against finite differences; the corrected route
1e-7). Sections 4c and 4d therefore read a response that was not a CCSD(T) Hessian; they stay as run. The probe now solves the (T) lambda explicitly, benzene
was recomputed the same evening (12 + 1 gradients; the curvature of E_CCSD(T) along three normal modes confirms the new Hessian to 2e-5, the old one was
7–9 % off out of plane), and the same test ran on it (`out/cc/A_8448043181_cc_test_analytic_tlambda*.json`).

In [11]:
tl_reg = json.load(open(OUT / "cc" / "A_8448043181_cc_test_analytic_tlambda.json")); tl_e = json.load(open(OUT / "cc" / "A_8448043181_cc_test_analytic_tlambda_all_band0.json"))
rows = []
for tag, rr in (("29 Sep, invalid CC response (CCSD lambda): registered band deck", an_reg), ("30 Sep, corrected CC response ((T) lambda): registered band deck", tl_reg),
                ("29 Sep, invalid CC response: wide pool, open prior", an_e), ("30 Sep, corrected CC response: wide pool, open prior", tl_e)):
    d = rr["cc"]
    rows.append({"run": tag, "P0 K_off(0.3)": d["P0"]["k_off_0p3"], "P1 median K_off ratio": d["P1_median"]["k_off_ratio"], "P1 median n_half ratio": d["P1_median"]["n_half_ratio"],
                 "oracle K_off(0.3)": d["oracle"]["k_off_0p3"], "in-band share CC": round(rr["delta2"]["inband_share_cc"], 2), "Frobenius CC/proxy": round(rr["delta2"]["ratio"], 2),
                 "C1": "pass" if rr["judged"]["C1_pass"] else "FAIL", "C2": "pass" if rr["judged"]["C2_pass"] else "FAIL"})
display(pd.DataFrame(rows).set_index("run"))

,P0 K_off(0.3),P1 median K_off ratio,P1 median n_half ratio,oracle K_off(0.3),in-band share CC,Frobenius CC/proxy,C1,C2
run,,,,,,,,
"29 Sep, invalid CC response (CCSD lambda): registered band deck",NaN,NaN,1.173913,NaN,0.04,2.36,FAIL,FAIL
"30 Sep, corrected CC response ((T) lambda): registered band deck",NaN,NaN,1.173913,NaN,0.10,2.68,FAIL,FAIL
"29 Sep, invalid CC response: wide pool, open prior",806.0,0.461538,0.571429,124.0,0.04,2.36,pass,FAIL
"30 Sep, corrected CC response: wide pool, open prior",806.0,0.538462,0.571429,124.0,0.10,2.68,pass,pass


*Reading (30 September 2026).* The corrected CC correction has 10 % of its off-diagonal power in band (not 4 %) and is 2.7× the proxy off the diagonal;
on the registered band deck no ordering reaches ρ_off 0.3, so C1 and C2 fail as before. On the wide pool with the open prior the band-trained scorer
now costs 0.54 of P0 (not 0.46; oracle 124 unchanged) and passes both C1 and C2 — the label for the plan line becomes "confirmed on this molecule (CC)"
for the wide deck. Directions unchanged, numbers replaced; the invalid response was systematically too soft out of plane, which is where the band deck
lives. Every CC-level read of this module from here uses the (T)-lambda Hessians only.

### 4g. Follow-up (4 October 2026): naphthalene on CC — the second molecule, and the requirement that was wrong

Naphthalene's CCSD(T)/cc-pVDZ Hessian with the explicit (T) lambda (1 October) is the second real response; the test ran at 02:3x–02:5x on 4 October
with the analytic B3LYP low level, the same deck and hash (`cc_level_test.py A_01f3186607 … --use-analytic --tag analytic_tlambda`, and the wide pool
with the open prior as the second cell). Both reads beside benzene's (4f):

In [12]:
na_reg = json.load(open(OUT / "cc" / "A_01f3186607_cc_test_analytic_analytic_tlambda.json")); na_e = json.load(open(OUT / "cc" / "A_01f3186607_cc_test_analytic_analytic_tlambda_all_band0.json"))
rows = []
for tag, rr in (("benzene, registered band deck (4f)", tl_reg), ("benzene, wide pool, open prior (4f)", tl_e),
                ("naphthalene, registered band deck", na_reg), ("naphthalene, wide pool, open prior", na_e)):
    for resp in ("cc", "proxy"):
        d = rr[resp]
        rows.append({"run": tag, "response": resp, "P0 K_off(0.3)": d["P0"]["k_off_0p3"], "P0 final ρ_off": round(d["P0"]["rho_off_final"], 2), "P1 median K_off ratio": d["P1_median"]["k_off_ratio"],
                     "P1 median n_half ratio": d["P1_median"]["n_half_ratio"], "oracle K_off(0.3)": d["oracle"]["k_off_0p3"]})
display(pd.DataFrame(rows))
print(f"naphthalene: in-band share of the off-diagonal Δ₂ power (200 cm⁻¹): CC {na_reg['delta2']['inband_share_cc']:.2f} vs proxy {na_reg['delta2']['inband_share_proxy']:.2f}; "
      f"Frobenius CC/proxy {na_reg['delta2']['ratio']:.2f}; band deck: C1 {'pass' if na_reg['judged']['C1_pass'] else 'FAIL'}, C2 {'pass' if na_reg['judged']['C2_pass'] else 'FAIL'}; "
      f"wide pool, open prior: C1 {'pass' if na_e['judged']['C1_pass'] else 'FAIL'}, C2 {'pass' if na_e['judged']['C2_pass'] else 'FAIL'}, "
      f"P1/oracle {na_e['judged']['C3']['p1_over_oracle']} ({na_e['judged']['label']})")

,run,response,P0 K_off(0.3),P0 final ρ_off,P1 median K_off ratio,P1 median n_half ratio,oracle K_off(0.3)
0,"benzene, registered band deck (4f)",cc,NaN,0.61,NaN,1.173913,NaN
1,"benzene, registered band deck (4f)",proxy,NaN,0.80,NaN,1.130435,NaN
2,"benzene, wide pool, open prior (4f)",cc,806.0,0.01,0.538462,0.571429,124.0
3,"benzene, wide pool, open prior (4f)",proxy,620.0,0.06,0.700000,0.555556,124.0
4,"naphthalene, registered band deck",cc,NaN,0.63,NaN,0.192308,NaN
5,"naphthalene, registered band deck",proxy,NaN,0.58,NaN,0.090909,NaN
6,"naphthalene, wide pool, open prior",cc,1716.0,0.01,0.454545,0.666667,156.0
7,"naphthalene, wide pool, open prior",proxy,1872.0,0.03,0.583333,0.111111,312.0


naphthalene: in-band share of the off-diagonal Δ₂ power (200 cm⁻¹): CC 0.30 vs proxy 0.28; Frobenius CC/proxy 2.74; band deck: C1 FAIL, C2 FAIL; wide pool, open prior: C1 pass, C2 pass, P1/oracle 5.0 (confirmed on this molecule (CC))


*Reading (4 October 2026).* The same picture as benzene, on the second molecule: on the band deck the registered lines fail; on the wide pool with the
open prior the scorer's order passes C1 and C2 and sits at five times the oracle's cost. Two molecules with one picture is no longer an accident, and the
user's reading of it changed the requirement rather than the method: the proposer orders measurements by what they teach about the *whole* correction,
which is what the network learns; judging it inside one spectral band measures a different and noisier thing (a handful of modes) and asks the method to
optimise a sub-problem. The in-band line had come from the Ladder's focus on the astronomical bands and had been carried into the CC test without that
argument being made. From 4 October the band-free read is the registered C1–C3 line (dated amendment in the pre-registration); the in-band numbers stay
in the records as what they are. Under that line both CC molecules are 'confirmed', and gate B's second molecule is passed. What this does not say: anything
about the learned representation (the scorer is the hand-feature one), or about molecules larger than naphthalene — anthracene's Hessian is the next.

## 5. The deck itself — the band candidate set against every pair

`deck_cost_readout.py` compares, per evaluation split, what the band deck and the all-pairs candidate set cost and buy. This is the result that changes
plan 05's deck design (pre-registered for after the 28th: `PreRegistration_2026-09-27_Wide_Candidate_Deck_Stop_Rule.md`).

In [13]:
dc = json.load(open(OUT / "sim" / "deck_cost_2026-09-27.json"))
rows = []
for split, r in dc.items():
    rows.append(dict(split=split, molecules=r["n_molecules"], band_deck_energies=r["band_deck_energies"], all_pairs_energies=r["wide_pool_energies"],
                     band_final_rho_off=round(r["band_deck_final_rho_off"], 3), all_pairs_final_rho_off=round(r["wide_pool_final_rho_off"], 3),
                     to_0p3_P0=r["wide_K_off_0p3_P0"], to_0p3_P1=r["wide_K_off_0p3_P1"], to_0p3_P12=r["wide_K_off_0p3_P12"], to_0p3_oracle=r["wide_K_off_0p3_oracle"],
                     P12_over_band_deck=round(r["wide_K_off_0p3_P12_over_band_deck"], 2)))
deck = pd.DataFrame(rows).set_index("split")
display(deck)

,molecules,band_deck_energies,all_pairs_energies,band_final_rho_off,all_pairs_final_rho_off,to_0p3_P0,to_0p3_P1,to_0p3_P12,to_0p3_oracle,P12_over_band_deck
split,,,,,,,,,,
eval_parents,41,1360.0,6614.0,0.530,0.076,2951.0,2232.0,1831.0,608.0,1.35
eval,56,1686.0,8448.0,0.471,0.142,3528.0,2928.0,2560.0,1120.0,1.52


## 6. Ethics and responsible use

The module proposes *which* calculation to run next, never a molecule or a claim about it. Its failure mode is wasted compute, not a wrong spectrum: the
stopping quantity (held-out ρ_off) is measured, not predicted. Every number is proxy-level (DFT against DFT) and labelled so; the oracle column is a ceiling,
not an achievable order. The data are the project's own corpus. The fair-chance rule protects the learned representation from a premature negative verdict
and, symmetrically, the reader from a premature positive one: nothing about it is licensed before stage 5.

## 7. Summary

In [14]:
e1p = table("band_p2_readout.json", "eval_parents"); e1s = table("band_p2_readout.json", "eval")
e2p = table("all_p2_readout.json", "eval_parents"); e2s = table("all_p2_readout.json", "eval")
best = lambda d, pref: float(min(d.loc[[i for i in d.index if i.startswith(pref)], "median_n_half_ratio"]))  # noqa: E731
summary = (f"On the band pool the learned order reaches the halfway point with {best(e1p, 'P1_'):.2f} (parents) / {best(e1s, 'P1_'):.2f} (substituted) of the fixed order's energies "
           f"(oracle {best(e1p, 'P3'):.2f} / {best(e1s, 'P3'):.2f}); on the all-pairs pool the fixed order itself ends at ρ_off {dc['eval_parents']['wide_pool_final_rho_off']:.2f} where the band deck ends at "
           f"{dc['eval_parents']['band_deck_final_rho_off']:.2f}, and the learned order reaches 0.3 at {dc['eval_parents']['wide_K_off_0p3_P12_over_band_deck']:.2f}× the whole band deck "
           f"(oracle {dc['eval_parents']['wide_K_off_0p3_oracle_over_band_deck']:.2f}×). Feedback during measurement helps the blind order and not the scorer's halfway point; "
           f"the gap to the oracle is knowledge. Stages 1–3 of the fair-chance search moved the validation Spearman from "
           f"{val.set_index('recipe').loc['p2', 'spearman'] if 'p2' in val.recipe.values else float('nan'):.3f} to {val.spearman.max():.3f} (P1 {val.set_index('recipe').loc['P1', 'spearman']:.3f}); nothing about the learned representation is licensed before stage 5.")
print(summary)
out = dict(date=time.strftime("%Y-%m-%d %H:%M"), molecules=len(df), splits=tab.to_dict(), live_molecule=dict(id=mid, M=int(e["M"]), curves={k: dict(final_rho_off=c[-1][2], k_off_0p3=C.k_off_at(c, 0.3, e["M"])) for k, c in curves.items()}),
           validation=val.to_dict(orient="records"), n_half_ratios=sm.to_dict(orient="records"), paired_adaptive=paired, paired_adaptive_wide=paired_w,
           deck_cost=dc, summary=summary)
json.dump(out, open("results.json", "w"), indent=1); print("results.json written")

On the band pool the learned order reaches the halfway point with 0.17 (parents) / 0.17 (substituted) of the fixed order's energies (oracle 0.06 / 0.07); on the all-pairs pool the fixed order itself ends at ρ_off 0.08 where the band deck ends at 0.53, and the learned order reaches 0.3 at 1.35× the whole band deck (oracle 0.45×). Feedback during measurement helps the blind order and not the scorer's halfway point; the gap to the oracle is knowledge. Stages 1–3 of the fair-chance search moved the validation Spearman from 0.611 to 0.648 (P1 0.582); nothing about the learned representation is licensed before stage 5.
results.json written
